In [2]:
%cd /content/drive/MyDrive/Colab\ Notebooks/ug_research

!git clone https://github.com/jeya-maria-jose/TransWeather.git
%cd TransWeather

/content/drive/MyDrive/Colab Notebooks/ug_research
fatal: destination path 'TransWeather' already exists and is not an empty directory.
/content/drive/MyDrive/Colab Notebooks/ug_research/TransWeather


In [3]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

from transweather_model import Transweather

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


# dataset loader

In [4]:
class FogDataset(Dataset):
  def __init__(self, fog_dir, clear_dir, transform=None):
    self.fog_dir = fog_dir
    self.clear_dir = clear_dir
    self.images = sorted(os.listdir(fog_dir))
    self.transform = transform

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    img_name = self.images[idx]

    fog = Image.open(os.path.join(self.fog_dir, img_name)).convert("RGB")
    clear = Image.open(os.path.join(self.clear_dir, img_name)).convert("RGB")

    if self.transform:
      fog = self.transform(fog)
      clear = self.transform(clear)

    return fog, clear

In [33]:
transform = transforms.Compose([
  transforms.Resize((256,256)),
  transforms.ToTensor()
])

In [34]:
train_dataset = FogDataset(
    os.path.join("/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/train", "fog"),
    os.path.join("/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/train", "clear"),
    transform
)

val_dataset = FogDataset(
    os.path.join("/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/val", "fog"),
    os.path.join("/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/val", "clear"),
    transform
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# model implementation

In [35]:
model = Transweather().to(torch.device("cuda"))

l1_loss = nn.L1Loss()
mse_loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# training

In [36]:
best_loss = float("inf")

for epoch in range(40):
  model.train()
  train_loss = 0

  for fog, clear in tqdm(train_loader):
    fog = fog.to(torch.device("cuda"))
    clear = clear.to(torch.device("cuda"))
    output = model(fog)
    output = torch.clamp(output, 0, 1)
    loss_l1 = l1_loss(output, clear)
    loss_mse = mse_loss(output, clear)
    loss = loss_l1 + 0.05 * loss_mse
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_loss += loss.item()

  print(f"Epoch {epoch+1}, Train Loss: {train_loss/len(train_loader)}")

  # Validation
  model.eval()
  val_loss = 0

  with torch.no_grad():
    for fog, clear in val_loader:
      fog = fog.to(torch.device("cuda"))
      clear = clear.to(torch.device("cuda"))

      output = model(fog)
      output = torch.clamp(output, 0, 1)

      loss_l1 = l1_loss(output, clear)
      loss_mse = mse_loss(output, clear)

      loss = loss_l1 + 0.05 * loss_mse

      val_loss += loss.item()

  val_loss /= len(val_loader)
  print(f"Val Loss: {val_loss}")

  # Save best
  if val_loss < best_loss:
    best_loss = val_loss
    torch.save(model.state_dict(), '/content/drive/MyDrive/Colab Notebooks/ug_research/transweather_fnt.pth')
    print("Best model saved!")

100%|██████████| 197/197 [00:45<00:00,  4.37it/s]


Epoch 1, Train Loss: 0.1448464894733453
Val Loss: 0.0988999636039226
Best model saved!


100%|██████████| 197/197 [00:46<00:00,  4.25it/s]


Epoch 2, Train Loss: 0.08616166550495903
Val Loss: 0.0643809971636569
Best model saved!


100%|██████████| 197/197 [00:47<00:00,  4.13it/s]


...

Epoch 38, Train Loss: 0.01807911057724868
Val Loss: 0.042120524841917334


100%|██████████| 197/197 [00:43<00:00,  4.48it/s]


Epoch 39, Train Loss: 0.018333804722742986
Val Loss: 0.042646970170048565


100%|██████████| 197/197 [00:43<00:00,  4.50it/s]


Epoch 40, Train Loss: 0.01802342632303232
Val Loss: 0.041903334774473715


# loading best model

In [37]:
model = Transweather().to(torch.device("cuda"))
model.load_state_dict(torch.load('/content/drive/MyDrive/Colab Notebooks/ug_research/transweather_fnt.pth', map_location=torch.device("cuda")))
model.eval()

Transweather(
  (Tenc): Tenc(
    (patch_embed1): OverlapPatchEmbed(
      (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (patch_embed2): OverlapPatchEmbed(
      (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (patch_embed3): OverlapPatchEmbed(
      (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
    )
    (patch_embed4): OverlapPatchEmbed(
      (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (mini_patch_embed1): OverlapPatchEmbed(
      (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  

# testing

In [38]:
def save_image(tensor, path):
  img = tensor.squeeze().permute(1,2,0).cpu().numpy()
  img = (img * 255).astype("uint8")
  Image.fromarray(img).save(path)

In [41]:
transform_test = transforms.Compose([
  transforms.Resize((256,256)),
  transforms.ToTensor()
])

OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/ug_research/TransWeather/outputs"
REAL_DIR = '/content/drive/MyDrive/Colab Notebooks/ug_research/real_frames'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for folder in sorted(os.listdir(REAL_DIR)):
  folder_path = os.path.join(REAL_DIR, folder)

  if not os.path.isdir(folder_path):
    continue

  save_folder = os.path.join(OUTPUT_DIR, folder)
  os.makedirs(save_folder, exist_ok=True)

  for img_name in sorted(os.listdir(folder_path)):
    if not img_name.lower().endswith((".png", ".jpg", ".jpeg")):
      continue

    img_path = os.path.join(folder_path, img_name)

    img = Image.open(img_path).convert("RGB")
    img = transform_test(img).unsqueeze(0).to(torch.device("cuda"))

    with torch.no_grad():
      output = model(img)
      output = torch.clamp(output, 0, 1)

    save_image(output, os.path.join(save_folder, img_name))

# evaluating and saving result

In [43]:
import cv2

input_root = "/content/drive/MyDrive/Colab Notebooks/ug_research/real_frames"
output_root = "/content/drive/MyDrive/Colab Notebooks/ug_research/TransWeather/outputs"

results = []

for i in range(1, 11):
  input_dir = os.path.join(input_root, f"rf_ds{i}")
  output_dir = os.path.join(output_root, f"rf_ds{i}")

  sharp_input = []
  sharp_output = []
  contrast_input = []
  contrast_output = []

  img_list = [f for f in os.listdir(input_dir) if f.endswith(".jpg")]

  for name in img_list:
    in_path = os.path.join(input_dir, name)
    out_path = os.path.join(output_dir, name)
    if not os.path.exists(out_path):
        continue

    inp = cv2.imread(in_path, 0)
    out = cv2.imread(out_path, 0)
    if inp is None or out is None:
        continue

    # Sharpness
    s1 = cv2.Laplacian(inp, cv2.CV_64F).var()
    s2 = cv2.Laplacian(out, cv2.CV_64F).var()
    sharp_input.append(s1)
    sharp_output.append(s2)

    # Contrast
    c1 = np.std(inp)
    c2 = np.std(out)
    contrast_input.append(c1)
    contrast_output.append(c2)

  if len(sharp_input) == 0:
      print(f"No valid images in rf_ds{i}")
      continue

  # Averages
  s_in = np.mean(sharp_input)
  s_out = np.mean(sharp_output)

  c_in = np.mean(contrast_input)
  c_out = np.mean(contrast_output)

  print(f"\nrf_ds{i}")
  print(f"Sharpness: {s_in:.2f} -> {s_out:.2f}")
  print(f"Contrast : {c_in:.2f} -> {c_out:.2f}")

  results.append([i, s_in, s_out, c_in, c_out])


rf_ds1
Sharpness: 99.65 -> 75.62
Contrast : 62.92 -> 52.67

rf_ds2
Sharpness: 168.15 -> 89.39
Contrast : 59.65 -> 52.53

rf_ds3
Sharpness: 84.43 -> 43.76
Contrast : 62.15 -> 50.11

rf_ds4
Sharpness: 25.67 -> 41.73
Contrast : 56.60 -> 49.98

rf_ds5
Sharpness: 242.72 -> 103.70
Contrast : 57.20 -> 54.46

rf_ds6
Sharpness: 278.10 -> 149.83
Contrast : 59.71 -> 51.36

rf_ds7
Sharpness: 54.05 -> 39.84
Contrast : 47.32 -> 46.71

rf_ds8
Sharpness: 58.27 -> 61.73
Contrast : 62.32 -> 53.99

rf_ds9
Sharpness: 32.16 -> 34.55
Contrast : 48.10 -> 45.73

rf_ds10
Sharpness: 175.72 -> 75.59
Contrast : 66.75 -> 53.73


In [44]:
# saving result
df = pd.DataFrame(results, columns=[
    "Dataset", "Sharp_input", "Sharp_output",
    "Contrast_input", "Contrast_output"
])

df.to_csv("/content/drive/MyDrive/Colab Notebooks/tw_real_ds_metrics.csv", index=False)